# 07 Time Series and Forecasting — Exercises

Practice time series analysis and short-term forecasting with the Songbai Nursing Home Legionnaires' disease line list.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import mean_absolute_error

# -- CJK font setup (prevents Chinese labels from showing as tofu boxes) --
# Scan the system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
cases = df[df["infected"] == 1]

## Question 1: Build a daily hospitalization-count series

1. Build a daily hospitalization-count time series using `hospitalization_date`
2. Fill in dates with no hospitalizations (`fill_value=0`)
3. Print the series length, date range, and total hospitalizations
4. Plot the hospitalization curve (bar chart) + a 5-day rolling average

In [ ]:
# TODO: build the daily hospitalization-count series
# TODO: fill in dates with no hospitalizations
# TODO: print the summary
# TODO: plot the hospitalization curve + 5-day rolling average

## Question 2: Hospitalization forecasting and window comparison

1. Forecast daily hospitalizations with 3-day, 5-day, and 7-day rolling averages (remember `shift(1)`)
2. Compute the MAE for each window
3. Which window performs best?
4. Plot Actual vs Predicted using the best window

In [ ]:
# TODO: compare MAE for 3 / 5 / 7-day windows
# TODO: pick the best window
# TODO: plot Actual vs Predicted

## Question 3 (challenge): Epidemic curves grouped by severity

1. Group cases by `clinical_severity` (mild / moderate / severe)
2. Build a daily onset-count series for each group
3. Plot the three severity epidemic curves with a **stacked bar chart**
4. Observe: are severe cases concentrated in a particular time window?

In [ ]:
# TODO: build daily series grouped by severity
# TODO: stacked bar chart
# TODO: observe and interpret

## Question 4: Seasonal pattern of weekly enterovirus reports (enterovirus scenario)

1. Build a daily report-count series using `report_date` (`asfreq("D", fill_value=0)`), then aggregate to **weekly** report counts with `resample("W").sum()`
2. Compute the average weekly report count for each month, and find the month with the highest reports
3. Build a "previous week" report count with `shift(1)`, and compute the week-over-week change (this week − previous week)
4. Plot the weekly epidemic curve, marking the weeks with the highest report counts
5. Interpret: how many peaks does the seasonal pattern of enterovirus show? Which childhood group activities might they be related to?

In [ ]:
import numpy as np

# --- Data: 2-year Enterovirus report line list ---
rng = np.random.default_rng(701)
start = pd.Timestamp("2024-01-01")
n_days = 730
dates_all = pd.date_range(start, periods=n_days, freq="D")
doy = np.array([d.dayofyear for d in dates_all]) % 366

# Two seasonal peaks: early summer (around April) and the start of school (around Sept-Oct)
peak1 = np.exp(-0.5 * ((doy - 100) / 22) ** 2)
peak2 = np.exp(-0.5 * ((doy - 270) / 28) ** 2)
weights = 0.05 + peak1 + 0.7 * peak2
probs = weights / weights.sum()

n_cases = 480
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
    "age_group": rng.choice(["<5", "5-9", "10-14"], size=n_cases, p=[0.55, 0.30, 0.15]),
}).sort_values("report_date").reset_index(drop=True)

# TODO: build the daily series using report_date (asfreq), then aggregate to weekly report counts with resample("W").sum()
# TODO: compute the average weekly report count per month, find the peak month
# TODO: use shift(1) to compute the week-over-week change
# TODO: plot the weekly epidemic curve, marking the weeks with the highest report counts
# TODO: interpret the seasonal pattern of enterovirus and its possible causes

## Question 5: SARIMA seasonal forecasting of daily influenza reports (influenza scenario)

1. Build a daily report-count time series (fill in missing days)
2. Test whether the series is stationary using `adfuller()`
3. Split into training / test sets (the last 14 days as the test set)
4. Fit `SARIMAX(order=(1,1,1), seasonal_order=(1,1,0,7))` and forecast daily reports over the test period
5. Compute the MAE of the SARIMA model and compare it against the simple baseline of "the last 7-day rolling average of the training set"
6. Interpret: does SARIMA capture the "lower weekend reporting" cyclical pattern better than the rolling-average baseline?

In [ ]:
import numpy as np

# --- Data: 120-day Influenza report line list ---
rng = np.random.default_rng(707)
start = pd.Timestamp("2025-11-01")
n_days = 120
dates_all = pd.date_range(start, periods=n_days, freq="D")
day_idx = np.arange(n_days)

# A single flu season: peaks around the midpoint (around day 60), plus a pattern of lower weekend reporting
season_shape = np.exp(-0.5 * ((day_idx - 60) / 18) ** 2)
dow_weight = np.where(dates_all.dayofweek < 5, 1.0, 0.45)  # fewer visits/reports on weekends
weights = (0.08 + season_shape) * dow_weight
probs = weights / weights.sum()

n_cases = 560
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
}).sort_values("report_date").reset_index(drop=True)

# TODO: build the daily report-count series (asfreq("D", fill_value=0))
# TODO: test stationarity using adfuller()
# TODO: split into training / test sets (last 14 days as the test set)
# TODO: fit SARIMAX(order=(1,1,1), seasonal_order=(1,1,0,7)) and forecast the test period
# TODO: compute SARIMA's MAE and compare it against the last-7-day rolling average baseline of the training set
# TODO: interpret whether SARIMA effectively captures the lower-weekend-reporting cyclical pattern

## Question 6: COVID-19 daily new cases and rolling-average smoothing (COVID-19 scenario)

1. Build a daily new-case-count series using `report_date` (fill in missing days)
2. Compute 3-day, 7-day, and 14-day rolling averages, and use `shift(1)` as "predicting the previous day's observation" to compute the MAE for each window
3. Find the window with the smallest MAE
4. Find the peak day of the "raw daily counts" and the "7-day rolling average" separately, and compare whether they agree
5. Interpret: why isn't the raw daily new-case count suitable for directly reading off the peak day of an epidemic curve?

In [ ]:
import numpy as np

# --- Data: 90-day COVID-19 report line list ---
rng = np.random.default_rng(719)
start = pd.Timestamp("2026-03-01")
n_days = 90
dates_all = pd.date_range(start, periods=n_days, freq="D")
day_idx = np.arange(n_days)

# A single wave: rises then falls, peaking around day 40, with "lower weekend reporting" noise layered on top
wave_shape = np.exp(-0.5 * ((day_idx - 40) / 14) ** 2)
dow_weight = np.where(dates_all.dayofweek < 5, 1.0, 0.5)
weights = (0.05 + wave_shape) * dow_weight
probs = weights / weights.sum()

n_cases = 600
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
}).sort_values("report_date").reset_index(drop=True)

# TODO: build the daily new-case-count series (asfreq("D", fill_value=0))
# TODO: compute 3 / 7 / 14-day rolling averages, use shift(1) to compute MAE and find the best window
# TODO: find the peak day of the raw daily counts and of the 7-day rolling average
# TODO: plot the daily new-case bar chart + 7-day rolling-average line
# TODO: interpret why the raw daily counts aren't suitable for directly reading off the peak day

## Question 7: Poisson / Negative Binomial regression for the dengue summer peak (dengue scenario)

1. Build a daily report-count series using `report_date` (fill in missing days)
2. Build lag features: `lag_1` (previous day), `lag_2` (two days prior), `day_idx` (day-count trend)
3. Compute the dispersion ratio (variance / mean) to determine whether the data are overdispersed
4. Fit a Poisson regression `cases ~ lag_1 + lag_2 + day_idx`; if overdispersed, refit a Negative Binomial regression and compare AIC
5. Convert the Negative Binomial regression coefficients to IRR (`exp(coef)`), and interpret the meaning of `lag_1`
6. Interpret: the possible causes of the dengue summer peak, and why dengue data are often overdispersed

In [ ]:
import numpy as np

# --- Data: one-year Dengue report line list ---
rng = np.random.default_rng(709)
start = pd.Timestamp("2025-01-01")
n_days = 365
dates_all = pd.date_range(start, periods=n_days, freq="D")
doy = np.arange(n_days)

# Summer peak: roughly June-September (mosquito vector density rises with temperature and humidity)
summer_shape = np.exp(-0.5 * ((doy - 210) / 35) ** 2)
weights = 0.05 + summer_shape
probs = weights / weights.sum()

n_cases = 520
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
    "township": rng.choice(["A區", "B區", "C區"], size=n_cases, p=[0.5, 0.3, 0.2]),
}).sort_values("report_date").reset_index(drop=True)

# TODO: build the daily report-count series (asfreq("D", fill_value=0))
# TODO: build lag_1, lag_2, day_idx features (remember to dropna)
# TODO: compute dispersion ratio = variance / mean to determine overdispersion
# TODO: fit a Poisson regression, and switch to Negative Binomial depending on the dispersion
# TODO: convert the coefficients to IRR and interpret the meaning of lag_1
# TODO: interpret the causes of the dengue summer peak and its overdispersion

## Question 8 (challenge): Continuing the nursing home story — the early-warning challenge of hot-water-system recontamination (Legionnaires' disease scenario)

About a month after the Songbai Nursing Home Legionnaires' disease outbreak was declared over, the infection control team suspected that the biofilm in the hot-water system had not been fully removed, which could lead to "recontamination and a second outbreak." This question extends the daily-onset concept from the main dataset, simulating the full course of "first-wave outbreak + a quiet period + second-wave outbreak," to test how your forecasting toolkit performs under a real challenge.

1. Build the complete daily onset-count series (including the first wave, the quiet period, and the second wave; fill missing days with 0)
2. **Using only data from the first wave and the quiet period (the first 51 days)**, train three models:
   - (a) 3-day rolling average (take the last rolling-average value from the training set as a fixed forecast for every future day)
   - (b) Poisson regression + lag features (`lag_1`, `lag_2`, `day_idx`), extrapolated forward via iterative prediction
   - (c) ARIMA(1,1,1)
3. Use these three models to "forecast" daily onset counts from day 52 onward (covering the entire second wave), and compute the MAE of each model
4. Plot a comparison of the three models' predictions vs. the actual onset counts
5. Interpret: did all three models underestimate the second wave? What does this imply for hospital infection control and real-time surveillance systems?

In [ ]:
import numpy as np

# --- Data: Songbai Nursing Home "recontamination" extended scenario (first wave + quiet period + second wave) ---
rng = np.random.default_rng(713)
start = pd.Timestamp("2026-02-01")

# First wave: a 21-day outbreak similar to the main dataset, peaking around day 8
wave1_days = 21
wave1_shape = np.exp(-0.5 * ((np.arange(wave1_days) - 8) / 3.5) ** 2)
wave1_dates = pd.date_range(start, periods=wave1_days, freq="D")

# Second wave: the hot-water system is recontaminated 55 days later, smaller in scale and more concentrated
wave2_start = start + pd.Timedelta(days=55)
wave2_days = 15
wave2_shape = np.exp(-0.5 * ((np.arange(wave2_days) - 6) / 3) ** 2)
wave2_dates = pd.date_range(wave2_start, periods=wave2_days, freq="D")

def _sample_wave(dates_wave, shape, n, rng):
    probs = shape / shape.sum()
    idx = rng.choice(len(dates_wave), size=n, p=probs)
    return dates_wave[idx]

onset1 = _sample_wave(wave1_dates, wave1_shape, 92, rng)
onset2 = _sample_wave(wave2_dates, wave2_shape, 34, rng)
onset_all = np.concatenate([onset1, onset2])

line_list = pd.DataFrame({
    "case_id": np.arange(1, len(onset_all) + 1),
    "symptom_onset_date": onset_all,
}).sort_values("symptom_onset_date").reset_index(drop=True)

# TODO: build the complete daily onset-count series using symptom_onset_date (from the first day of wave 1 to the last day of wave 2, fill missing days with 0)
# TODO: split into a training set = the first 51 days (wave 1 + quiet period), test set = day 52 onward (covering wave 2)
# TODO: fit (a) 3-day rolling-average fixed value (b) Poisson + lag iterative prediction (c) ARIMA(1,1,1) on the training set, and forecast the test period with each
# TODO: compute the MAE for the three models, and plot predicted vs. actual
# TODO: interpret whether all three models underestimated the second wave, and what this implies for real-time surveillance and infection-control early warning